In [1]:
# 1. Installations and Dependencies

# Core Agents & AI
# %pip install -qU langchain-openai
# %pip install -qU langchain-community
# %pip install -qU langgraph
# %pip install -qU python-dotenv
# %pip install -qU tavily-python

# Web Scraping Tools
# %pip install -qU ddgs
# %pip install -qU selenium
# %pip install -qU webdriver-manager
# %pip install -qU beautifulsoup4

print("[✅] Dependencies configuration checked.")

[✅] Dependencies configuration checked.


In [2]:
# 2. Environment setup: Tavily Search Client
import os
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv()

# Verify API Key
tavily_api_key = os.getenv("TAVILY_API_KEY")
if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found in .env file.")

# Initialize Client
tavily_client = TavilyClient(tavily_api_key)

print("[✅] Tavily Client initialized.")

# Optional: Quick Connectivity Test
# try:
#     test_response = tavily_client.search(query="test connectivity", max_results=1)
#     if test_response and 'results' in test_response:
#         print("[✅] Tavily API connection successful.")
#     else:
#         print("[⚠️] Tavily connected but returned no results.")
# except Exception as e:
#     print(f"[❌] Tavily Connection failed: {e}")

[✅] Tavily Client initialized.


In [3]:
# 3. Environment setup: Local LLM (Llama 3 Power)
from langchain_openai import ChatOpenAI

# LM Studio Configuration
lm_studio_base = "http://localhost:1234/v1"
lm_studio_key = "lm-studio" 

# Initialize the LLM
# DICA: Certifique-se que o "Context Length" no LM Studio está setado para 8192 ou mais!
llm = ChatOpenAI(
    model="meta-llama-3.1-8b-instruct", # Atualizado para o Llama 3
    base_url=lm_studio_base,
    api_key=lm_studio_key,
    temperature=0
)

print(f"Target Model: meta-llama-3.1-8b-instruct at {lm_studio_base}")
try:
    response = llm.invoke("System check. Reply 'Online'.").content
    print(f"[✅] Local LLM Status: {response}")
except Exception as e:
    print(f"[❌] Local LLM Connection failed: {e}")

Target Model: meta-llama-3.1-8b-instruct at http://localhost:1234/v1
[✅] Local LLM Status: Online.


In [4]:
# 4. Task Definition (International Standard)

TARGET_CITY = "New York City"
TOPIC = "Pizza Places"

# Query para o LLM extrair os dados depois
EXTRACTION_QUERY = f"""
Analyze the HTML content provided about {TARGET_CITY}.
Identify the section regarding 'Eat', 'Food', or 'Restaurants'.
Extract 5 famous Pizzerias or Restaurants mentioned.
Return a JSON list with:
- Name
- Description/Specialty
- Price Range (if available)
- Location/Neighborhood

Output ONLY JSON.
"""

print(f"[✅] Task Defined: Find {TOPIC} in {TARGET_CITY}")

[✅] Task Defined: Find Pizza Places in New York City


In [5]:
# 5. Agentic Search: Hybrid (Neighborhood Mode + Tavily Fallback)
from ddgs import DDGS

print("="*60)
print("🔍 AGENTIC SEARCH (Hybrid Strategy)")
print("="*60)

# Focamos em um bairro específico (Little Italy) para garantir vcards de restaurantes
search_query = f"site:en.wikivoyage.org Manhattan Little Italy Eat"
target_url = None

# --- STEP 1: DuckDuckGo (Free) ---
print(f"1️⃣ Attempting DuckDuckGo Search for: '{search_query}'...")

try:
    with DDGS() as ddgs:
        results = list(ddgs.text(search_query, max_results=5))
        
        if results:
            print(f"   ↳ DDGS found {len(results)} candidates.")
            for r in results:
                url = r['href']
                print(f"     - Checking: {url}")
                
                # FILTRO DE QUALIDADE:
                # 1. Deve ser wikivoyage
                # 2. NÃO pode ser 'Category:' (índice sem conteúdo)
                # 3. NÃO pode ser 'File:' (imagem)
                if "wikivoyage.org" in url and "Category:" not in url and "File:" not in url:
                    target_url = url
                    print(f"   ✅ Verified Content Page: {url}")
                    break
                else:
                    print(f"     ❌ Rejected (Category/File/Ad)")
            
except Exception as e:
    print(f"   ❌ DDGS Error: {e}")

# --- STEP 2: Tavily (Fallback) ---
if not target_url:
    print(f"\n2️⃣ DDGS failed/empty. Switching to Tavily Agent...")
    try:
        # Tavily é pago/limitado, mas muito bom em seguir intenção
        tavily_results = tavily_client.search(query=search_query, max_results=5)
        
        if tavily_results and "results" in tavily_results:
            print(f"   ↳ Tavily found {len(tavily_results['results'])} candidates.")
            for res in tavily_results["results"]:
                url = res["url"]
                print(f"     - Checking: {url}")
                
                # Mesmo filtro de qualidade
                if "wikivoyage.org" in url and "Category:" not in url:
                    target_url = url
                    print(f"   ✅ Tavily Verified Source: {url}")
                    break
                else:
                    print(f"     ❌ Rejected (Wrong Domain/Category)")
                    
    except Exception as e:
        print(f"   ❌ Tavily Error: {e}")

# --- STEP 3: Fail-Safe (Lesson Guarantee) ---
if target_url:
    print(f"\n[🎯] FINAL TARGET URL: {target_url}")
else:
    print(f"\n[⚠️] Both Agents failed to find a clean URL. Using Direct Fallback.")
    target_url = "https://en.wikivoyage.org/wiki/Manhattan/Little_Italy"
    print(f"[🎯] FINAL TARGET URL: {target_url}")

🔍 AGENTIC SEARCH (Hybrid Strategy)
1️⃣ Attempting DuckDuckGo Search for: 'site:en.wikivoyage.org Manhattan Little Italy Eat'...
   ↳ DDGS found 5 candidates.
     - Checking: https://en.wikivoyage.org/wiki/Little_Italy
   ✅ Verified Content Page: https://en.wikivoyage.org/wiki/Little_Italy

[🎯] FINAL TARGET URL: https://en.wikivoyage.org/wiki/Little_Italy


In [6]:
# 6. Web Scraper Definition (Simple Driver)
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

def scrape_page_content(url):
    if not url: return None
    driver = None
    soup = None
    try:
        print(f"   ↳ Initializing Driver...")
        service = Service(ChromeDriverManager().install())
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
    
        driver = webdriver.Chrome(service=service, options=options)
        print(f"   ↳ Loading: {url}...")
        driver.get(url)
        time.sleep(3) # Wait for render
        
        page_source = driver.page_source
        if len(page_source) > 5000:
            soup = BeautifulSoup(page_source, 'html.parser')
            print(f"   ↳ Success! Downloaded {len(page_source)} chars.")
        else:
            print(f"   ⚠️ Content too small ({len(page_source)} chars).")

    except Exception as e:
        print(f"[❌] Error: {e}")
    finally:
        if driver: driver.quit()
            
    return soup

In [7]:
# 7. Execution: Scrape Target
tripadvisor_soup = None

# Garantia contra NameError
if 'TARGET_CITY' not in locals():
    TARGET_CITY = "New York City"

print(f"🕷️ SCRAPING TARGET: {TARGET_CITY}")
print("="*60)

if 'target_url' in locals() and target_url:
    tripadvisor_soup = scrape_page_content(target_url)
    
    if tripadvisor_soup:
        print("\n[✅] SCRAPING SUCCESS!")
        title = tripadvisor_soup.find('title').get_text(strip=True)
        print(f"Page Title: {title}")
    else:
        print("\n[❌] SCRAPING FAILED.")
else:
    print("\n[⚠️] ABORTED: No URL found in Cell 5.")

🕷️ SCRAPING TARGET: New York City
   ↳ Initializing Driver...
   ↳ Loading: https://en.wikivoyage.org/wiki/Little_Italy...
   ↳ Success! Downloaded 251412 chars.

[✅] SCRAPING SUCCESS!
Page Title: Little Italy – Travel guide at Wikivoyage


In [8]:
# 8. HTML Parsing (High Context Mode for Llama 3)
from bs4 import BeautifulSoup

print("="*60)
print("🥣 HTML PARSING (HIGH CONTEXT / VCARD)")
print("="*60)

if not tripadvisor_soup:
    raise ValueError("No data to parse.")

current_soup = tripadvisor_soup
raw_html_snippet = ""

# Estratégia Vcard (WikiVoyage)
vcards = current_soup.find_all(class_='vcard')

if vcards:
    print(f"[✅] Found {len(vcards)} listing cards (vcards).")
    print("   ↳ Compiling ALL restaurant data...")
    
    # AGORA SEM LIMITES RÍGIDOS DE CONTAGEM
    # O Llama 3 aguenta muito mais contexto.
    for card in vcards:
        card_text = str(card)
        # Filtro leve apenas para garantir que tem conteúdo
        if 'listing-content' in card_text or 'address' in card_text:
            raw_html_snippet += card_text + "\n"
            
            # Limite de segurança aumentado para ~8k tokens (aprox 35k chars)
            if len(raw_html_snippet) > 35000: 
                print("   ⚠️ Context limit (35k chars) reached. Stopping collection.")
                break

    # Fallback se os vcards estiverem vazios
    if len(raw_html_snippet) < 500:
        print("   ⚠️ Vcards empty. Dumping main body.")
        body = current_soup.find('div', class_='mw-parser-output')
        if body:
            raw_html_snippet = body.get_text(separator="\n")[:35000]

else:
    print("[⚠️] No 'vcard' found. Dumping raw body text.")
    body = current_soup.find('div', class_='mw-parser-output')
    if body:
        raw_html_snippet = body.get_text(separator="\n")[:35000]

top_n_restaurants = [raw_html_snippet]
print(f"[✂️] Context Ready: {len(raw_html_snippet)} characters.")
print("[🚀] Ready for Llama 3 Extraction.")

🥣 HTML PARSING (HIGH CONTEXT / VCARD)
[⚠️] No 'vcard' found. Dumping raw body text.
[✂️] Context Ready: 0 characters.
[🚀] Ready for Llama 3 Extraction.


In [9]:
# 9. Manual Extraction Check (Bypassed for Agentic Workflow)
print("="*60)
print("⏭️ MANUAL EXTRACTION STEP")
print("="*60)

# Verificação de segurança
if not top_n_restaurants or not isinstance(top_n_restaurants[0], str):
    print("[❌] Error: Data format incorrect. Expected raw HTML string from Cell 8.")
else:
    data_sample = top_n_restaurants[0][:500].replace("\n", " ")
    
    print("[ℹ️] STATUS UPDATE:")
    print("      We have switched targets from TripAdvisor to WikiVoyage.")
    print("      Manual CSS selectors (like '.tbrcR') are site-specific and brittle.")
    print("      Instead of rewriting CSS selectors manually, we will let the AI agent")
    print("      handle the structure in the next step.")
    print("-" * 60)
    print(f"[✅] Raw Data Ready for AI: {len(top_n_restaurants[0])} characters.")
    print(f"[👀] Preview: {data_sample}...")
    print("-" * 60)
    print("👉 Proceed to Cell 10 to configure the AI Prompt.")

⏭️ MANUAL EXTRACTION STEP
[ℹ️] STATUS UPDATE:
      We have switched targets from TripAdvisor to WikiVoyage.
      Manual CSS selectors (like '.tbrcR') are site-specific and brittle.
      Instead of rewriting CSS selectors manually, we will let the AI agent
      handle the structure in the next step.
------------------------------------------------------------
[✅] Raw Data Ready for AI: 0 characters.
[👀] Preview: ...
------------------------------------------------------------
👉 Proceed to Cell 10 to configure the AI Prompt.


In [10]:
# 10. Agentic Extraction Setup (Prompt Engineering)
from langchain_core.messages import HumanMessage, SystemMessage

print("="*60)
print("🧠 PREPARING LLM EXTRACTION")
print("="*60)

# 1. Prepare Data context
# We convert the BeautifulSoup object back to a string, 
# but specifically focusing on the restaurant blocks we found to save tokens.
# (If we sent the whole page, it might crash the local model context window)
raw_html_snippet = ""
for block in top_n_restaurants: # Reusing the blocks found in Cell 8
    raw_html_snippet += str(block) + "\n"

print(f"[📊] Data Context Size: {len(raw_html_snippet)} characters")

# 2. Define the Extraction Prompt
# Note: We use the 'EXTRACTION_QUERY' defined way back in Cell 4 as the goal.
extraction_prompt = f"""
TASK: {EXTRACTION_QUERY}

SOURCE HTML:
{raw_html_snippet}

INSTRUCTIONS:
1. Analyze the SOURCE HTML above.
2. Extract the data exactly as requested in the TASK.
3. If a piece of info is missing (like specific price), write "N/A".
4. Format your response as a clean Python List of Dictionaries or JSON.
5. Do NOT explain your process, just return the data.
"""

print("[✅] Prompt constructed. Sending to Local LLM...")
# Optional: Print preview of prompt
# print(f"Prompt Preview:\n{extraction_prompt[:300]}...")

🧠 PREPARING LLM EXTRACTION
[📊] Data Context Size: 1 characters
[✅] Prompt constructed. Sending to Local LLM...


In [11]:
# 11. Execution: LLM Data Extraction (Llama 3 Edition)
import json
import re

print("="*60)
print("🤖 AI AGENT EXTRACTION (Meta Llama 3.1 8B)")
print("="*60)

try:
    print(f"   ↳ Sending {len(extraction_prompt)} chars to model...")
    print("   ↳ Processing... (Llama 3.1 might take a bit longer but is smarter)")
    
    response = llm.invoke(extraction_prompt)
    content = response.content
    
    # Limpeza de Markdown (Llama 3.1 adora usar markdown)
    clean_content = re.sub(r"```json|```", "", content).strip()
    
    print("\n[✅] EXTRACTION COMPLETE!\n")
    print("-" * 40)
    print(clean_content)
    print("-" * 40)
    
    # Validação
    if "{" in clean_content and "}" in clean_content:
        print("\n[🎯] Validation: Structure looks correct.")
    else:
        print("\n[⚠️] Validation: Output might not be JSON. Check raw response.")

except Exception as e:
    print(f"[❌] LLM Extraction Failed: {e}")
    if "400" in str(e):
        print("\n🔴 CRITICAL: Context Overflow.")
        print("👉 ACTION: Go to LM Studio -> Right Sidebar -> Increase 'Context Length' to 8192.")

print("="*60)

🤖 AI AGENT EXTRACTION (Meta Llama 3.1 8B)
   ↳ Sending 623 chars to model...
   ↳ Processing... (Llama 3.1 might take a bit longer but is smarter)

[✅] EXTRACTION COMPLETE!

----------------------------------------
[
  {
    "Name": "Joe's Pizza",
    "Description/Specialty": "Classic New York-style pizza",
    "Price Range": "$10-$20",
    "Location/Neighborhood": "Greenwich Village"
  },
  {
    "Name": "Lombardi's",
    "Description/Specialty": "First pizzeria in the United States, classic Neapolitan-style pizza",
    "Price Range": "$15-$30",
    "Location/Neighborhood": "Little Italy"
  },
  {
    "Name": "Grimaldi's",
    "Description/Specialty": "Coal-fired brick oven, classic Neapolitan-style pizza",
    "Price Range": "$10-$25",
    "Location/Neighborhood": "Brooklyn Bridge Park"
  },
  {
    "Name": "Patsy's Pizzeria",
    "Description/Specialty": "Classic New York-style pizza, historic pizzeria",
    "Price Range": "$15-$30",
    "Location/Neighborhood": "East Harlem"
  },
 